In [1]:
# ============================================================
# PERSONALIZED HEALTHCARE SYSTEM
# NOTEBOOK 03 - UNIFIED MODEL TRAINING
# Dataset 1 + Dataset 3
# ============================================================

import os
import re
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

RANDOM_STATE = 42

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATASET_1_PATH = os.path.join(
    BASE_DIR, "data", "dataset_1", "dataset.csv"
)

DATASET_3_PATH = os.path.join(
    BASE_DIR, "data", "dataset_3",
    "Diseases_and_Symptoms_dataset.csv"
)

MODEL_DIR = os.path.join(
    BASE_DIR, "models"
)

os.makedirs(MODEL_DIR, exist_ok=True)

print("=" * 80)
print("PERSONALIZED HEALTHCARE SYSTEM")
print("UNIFIED DISEASE PREDICTION MODEL")
print("=" * 80)

print("Base directory:", BASE_DIR)
print("Dataset 1:", DATASET_1_PATH)
print("Dataset 3:", DATASET_3_PATH)

PERSONALIZED HEALTHCARE SYSTEM
UNIFIED DISEASE PREDICTION MODEL
Base directory: c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System
Dataset 1: c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System\data\dataset_1\dataset.csv
Dataset 3: c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System\data\dataset_3\Diseases_and_Symptoms_dataset.csv


In [2]:
# ============================================================
# LOAD DATASETS
# ============================================================

if not os.path.exists(DATASET_1_PATH):
    raise FileNotFoundError(
        f"Dataset 1 not found:\n{DATASET_1_PATH}"
    )

if not os.path.exists(DATASET_3_PATH):
    raise FileNotFoundError(
        f"Dataset 3 not found:\n{DATASET_3_PATH}"
    )

df1 = pd.read_csv(DATASET_1_PATH)
df3 = pd.read_csv(DATASET_3_PATH)

print("=" * 80)
print("DATASETS LOADED")
print("=" * 80)

print("Dataset 1 shape:", df1.shape)
print("Dataset 3 shape:", df3.shape)

print("Dataset 1 diseases:", df1["Disease"].nunique())
print("Dataset 3 diseases:", df3["diseases"].nunique())

DATASETS LOADED
Dataset 1 shape: (4920, 18)
Dataset 3 shape: (96088, 231)
Dataset 1 diseases: 41
Dataset 3 diseases: 100


In [3]:
# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value).strip().lower()

    value = value.replace("_", " ")
    value = value.replace("-", " ")

    value = re.sub(r"\s+", " ", value)

    return value.strip()


def compact_text(value):

    value = normalize_text(value)

    return re.sub(
        r"[^a-z0-9]+",
        "",
        value
    )

In [4]:
# ============================================================
# DISEASE NORMALIZATION
# ============================================================

def normalize_disease(value):

    value = normalize_text(value)

    # Dataset 1 spelling / formatting corrections
    replacements = {
        "(vertigo) paroymsal  positional vertigo":
            "(vertigo) paroxysmal positional vertigo",

        "peptic ulcer diseae":
            "peptic ulcer disease",

        "osteoarthristis":
            "osteoarthritis",
    }

    return replacements.get(
        value,
        value
    )


df1["_disease_normalized"] = (
    df1["Disease"]
    .apply(normalize_disease)
)

df3["_disease_normalized"] = (
    df3["diseases"]
    .apply(normalize_disease)
)

diseases_1 = set(
    df1["_disease_normalized"]
)

diseases_3 = set(
    df3["_disease_normalized"]
)

common_diseases = diseases_1 & diseases_3

all_diseases = diseases_1 | diseases_3

print("=" * 80)
print("DISEASE ANALYSIS")
print("=" * 80)

print("Dataset 1 diseases:", len(diseases_1))
print("Dataset 3 diseases:", len(diseases_3))
print("Common diseases:", len(common_diseases))
print("Unified diseases:", len(all_diseases))

print("\nCommon diseases:")
for disease in sorted(common_diseases):
    print("-", disease)

DISEASE ANALYSIS
Dataset 1 diseases: 41
Dataset 3 diseases: 100
Common diseases: 8
Unified diseases: 133

Common diseases:
- allergy
- common cold
- drug reaction
- heart attack
- hypoglycemia
- pneumonia
- psoriasis
- urinary tract infection


In [5]:
# ============================================================
# DATASET 3 SYMPTOM VOCABULARY
# ============================================================

dataset3_symptom_columns = list(
    df3.columns[1:]
)

dataset3_normalized_features = {
    normalize_text(column): column
    for column in dataset3_symptom_columns
}

print("=" * 80)
print("DATASET 3 SYMPTOM VOCABULARY")
print("=" * 80)

print(
    "Dataset 3 symptom features:",
    len(dataset3_symptom_columns)
)

DATASET 3 SYMPTOM VOCABULARY
Dataset 3 symptom features: 231


In [6]:
# ============================================================
# DATASET 1 SYMPTOMS
# ============================================================

dataset1_symptom_columns = [
    column
    for column in df1.columns
    if str(column).lower().startswith("symptom")
]

dataset1_symptoms = set()

for column in dataset1_symptom_columns:

    for value in df1[column]:

        normalized = normalize_text(value)

        if normalized:
            dataset1_symptoms.add(normalized)


print("=" * 80)
print("DATASET 1 SYMPTOMS")
print("=" * 80)

print(
    "Dataset 1 symptom columns:",
    len(dataset1_symptom_columns)
)

print(
    "Unique Dataset 1 symptoms:",
    len(dataset1_symptoms)
)

DATASET 1 SYMPTOMS
Dataset 1 symptom columns: 17
Unique Dataset 1 symptoms: 131


In [7]:
# ============================================================
# SYMPTOM ALIAS / SEMANTIC NORMALIZATION
# ============================================================

SYMPTOM_ALIASES = {

    # Heart
    "fast heart rate":
        "increased heart rate",

    "rapid heart rate":
        "increased heart rate",

    "high heart rate":
        "increased heart rate",

    "heart rate increased":
        "increased heart rate",

    "palpitation":
        "palpitations",

    # Respiratory
    "breathlessness":
        "shortness of breath",

    "difficulty breathing":
        "difficulty breathing",

    "breathing difficulty":
        "difficulty breathing",

    # Skin
    "skin rashes":
        "skin rash",

    "rash":
        "skin rash",

    # Urinary
    "frequent urination":
        "frequent urination",

    "excess urination":
        "excessive urination",

    # Vomiting
    "vomit":
        "vomiting",

    # Head
    "head pain":
        "headache",

    # Joint
    "joint aches":
        "joint pain",
}


def normalize_symptom_name(value):

    value = normalize_text(value)

    if value in SYMPTOM_ALIASES:
        return SYMPTOM_ALIASES[value]

    return value

In [8]:
# ============================================================
# DATASET 1 -> DATASET 3 SYMPTOM MATCHING
# ============================================================

matched_symptoms = {}
unmatched_symptoms = []

for symptom in sorted(dataset1_symptoms):

    normalized = normalize_symptom_name(
        symptom
    )

    if normalized in dataset3_normalized_features:

        matched_symptoms[symptom] = (
            dataset3_normalized_features[
                normalized
            ]
        )

    else:

        unmatched_symptoms.append(
            symptom
        )


print("=" * 80)
print("SYMPTOM MATCHING")
print("=" * 80)

print(
    "Dataset 1 unique symptoms:",
    len(dataset1_symptoms)
)

print(
    "Matched to Dataset 3:",
    len(matched_symptoms)
)

print(
    "Still unmatched:",
    len(unmatched_symptoms)
)

print("\nMatched examples:")

for source, target in list(
    matched_symptoms.items()
)[:30]:

    print(
        f"{source}  -->  {target}"
    )

print("\nUnmatched examples:")

for symptom in unmatched_symptoms[:30]:

    print(
        symptom
    )

SYMPTOM MATCHING
Dataset 1 unique symptoms: 131
Matched to Dataset 3: 20
Still unmatched: 111

Matched examples:
back pain  -->  back pain
breathlessness  -->  shortness of breath
chills  -->  chills
constipation  -->  constipation
cough  -->  cough
depression  -->  depression
dizziness  -->  dizziness
fast heart rate  -->  increased heart rate
fatigue  -->  fatigue
headache  -->  headache
joint pain  -->  joint pain
knee pain  -->  knee pain
nausea  -->  nausea
neck pain  -->  neck pain
palpitations  -->  palpitations
restlessness  -->  restlessness
skin rash  -->  skin rash
sweating  -->  sweating
vomiting  -->  vomiting
weight gain  -->  weight gain

Unmatched examples:
abdominal pain
abnormal menstruation
acidity
acute liver failure
altered sensorium
anxiety
belly pain
blackheads
bladder discomfort
blister
blood in sputum
bloody stool
blurred and distorted vision
brittle nails
bruising
burning micturition
chest pain
cold hands and feets
coma
congestion
continuous feel of urine
cont

In [9]:
# ============================================================
# UNIFIED SYMPTOM VOCABULARY
# ============================================================

unified_features = set(
    dataset3_symptom_columns
)

# Dataset 1 ke unmatched symptoms ko bhi preserve karo
for symptom in unmatched_symptoms:

    unified_features.add(
        normalize_symptom_name(symptom)
    )


unified_features = sorted(
    unified_features
)

print("=" * 80)
print("UNIFIED FEATURE SPACE")
print("=" * 80)

print(
    "Dataset 3 original features:",
    len(dataset3_symptom_columns)
)

print(
    "Dataset 1 unmatched features added:",
    len(unmatched_symptoms)
)

print(
    "Final unified features:",
    len(unified_features)
)

UNIFIED FEATURE SPACE
Dataset 3 original features: 231
Dataset 1 unmatched features added: 111
Final unified features: 342


In [10]:
# ============================================================
# CONVERT DATASET 3 TO UNIFIED FORMAT
# ============================================================

X3 = pd.DataFrame(
    0,
    index=np.arange(len(df3)),
    columns=unified_features,
    dtype=np.int8
)

for original_column in dataset3_symptom_columns:

    normalized_column = normalize_symptom_name(
        original_column
    )

    if normalized_column in X3.columns:

        values = pd.to_numeric(
            df3[original_column],
            errors="coerce"
        ).fillna(0)

        X3[normalized_column] = (
            values.astype(np.int8)
            .clip(0, 1)
        )


y3 = df3[
    "_disease_normalized"
].reset_index(drop=True)

X3 = X3.reset_index(drop=True)

print("=" * 80)
print("DATASET 3 CONVERSION")
print("=" * 80)

print("X3:", X3.shape)
print("y3:", y3.shape)

DATASET 3 CONVERSION
X3: (96088, 342)
y3: (96088,)


In [11]:
# ============================================================
# CONVERT DATASET 1 TO UNIFIED FORMAT
# ============================================================

X1 = pd.DataFrame(
    0,
    index=np.arange(len(df1)),
    columns=unified_features,
    dtype=np.int8
)

for _, row in df1.iterrows():

    for column in dataset1_symptom_columns:

        symptom = normalize_symptom_name(
            row[column]
        )

        if not symptom:
            continue

        if symptom in X1.columns:

            X1.loc[
                _,
                symptom
            ] = 1


y1 = df1[
    "_disease_normalized"
].reset_index(drop=True)

X1 = X1.reset_index(drop=True)

print("=" * 80)
print("DATASET 1 CONVERSION")
print("=" * 80)

print("X1:", X1.shape)
print("y1:", y1.shape)

DATASET 1 CONVERSION
X1: (4920, 342)
y1: (4920,)


In [12]:
# ============================================================
# COMBINE DATASET 1 + DATASET 3
# ============================================================

X = pd.concat(
    [
        X1,
        X3
    ],
    axis=0,
    ignore_index=True
)

y = pd.concat(
    [
        y1,
        y3
    ],
    axis=0,
    ignore_index=True
)

print("=" * 80)
print("UNIFIED DATASET")
print("=" * 80)

print("X shape:", X.shape)
print("y shape:", y.shape)

print(
    "Total records:",
    len(X)
)

print(
    "Total diseases:",
    y.nunique()
)

print(
    "Total features:",
    X.shape[1]
)

UNIFIED DATASET
X shape: (101008, 342)
y shape: (101008,)
Total records: 101008
Total diseases: 133
Total features: 342


In [13]:
# ============================================================
# UNIFIED DATASET VALIDATION
# ============================================================

print("=" * 80)
print("UNIFIED DATASET VALIDATION")
print("=" * 80)

print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

unique_values = set(
    X.to_numpy().flatten()
)

print(
    "Feature values:",
    sorted(unique_values)
)

if not unique_values.issubset({0, 1}):
    raise ValueError(
        "Unified feature matrix contains values other than 0 and 1."
    )

class_counts = y.value_counts()

print(
    "\nMinimum records for any disease:",
    class_counts.min()
)

if (class_counts < 2).any():

    print(
        class_counts[
            class_counts < 2
        ]
    )

    raise ValueError(
        "Some disease classes have fewer than 2 records."
    )

print("\nValidation PASSED.")

UNIFIED DATASET VALIDATION
Missing values in X: 0
Missing values in y: 0
Feature values: [np.int8(0), np.int8(1)]

Minimum records for any disease: 120

Validation PASSED.


In [14]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=RANDOM_STATE,

    stratify=y
)

print("=" * 80)
print("TRAIN / TEST SPLIT")
print("=" * 80)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("Training features:", X_train.shape[1])
print("Testing features:", X_test.shape[1])

print("Training diseases:", y_train.nunique())
print("Testing diseases:", y_test.nunique())

TRAIN / TEST SPLIT
Training samples: 80806
Testing samples: 20202
Training features: 342
Testing features: 342
Training diseases: 133
Testing diseases: 133


In [15]:
# ============================================================
# LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

y_train_encoded = (
    label_encoder.fit_transform(
        y_train
    )
)

y_test_encoded = (
    label_encoder.transform(
        y_test
    )
)

print("=" * 80)
print("LABEL ENCODING")
print("=" * 80)

print(
    "Number of classes:",
    len(label_encoder.classes_)
)

for index, disease in enumerate(
    label_encoder.classes_,
    start=1
):

    print(
        f"{index:03d}. {disease}"
    )

LABEL ENCODING
Number of classes: 133
001. (vertigo) paroymsal positional vertigo
002. acne
003. actinic keratosis
004. acute bronchiolitis
005. acute bronchitis
006. acute bronchospasm
007. acute kidney injury
008. acute pancreatitis
009. acute sinusitis
010. aids
011. alcoholic hepatitis
012. allergy
013. angina
014. anxiety
015. appendicitis
016. arthritis
017. arthritis of the hip
018. asthma
019. benign prostatic hyperplasia (bph)
020. brachial neuritis
021. bronchial asthma
022. bursitis
023. carpal tunnel syndrome
024. cervical spondylosis
025. chicken pox
026. cholecystitis
027. chronic back pain
028. chronic cholestasis
029. chronic constipation
030. chronic obstructive pulmonary disease (copd)
031. common cold
032. complex regional pain syndrome
033. concussion
034. conjunctivitis
035. conjunctivitis due to allergy
036. contact dermatitis
037. cornea infection
038. croup
039. cystitis
040. degenerative disc disease
041. dengue
042. dental caries
043. depression
044. developme

In [16]:
# ============================================================
# RANDOM FOREST
# ============================================================
model = RandomForestClassifier(
    n_estimators=30,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)



print("=" * 80)
print("TRAINING RANDOM FOREST")
print("=" * 80)

model.fit(
    X_train,
    y_train_encoded
)

print("\nTraining completed successfully.")

TRAINING RANDOM FOREST

Training completed successfully.


In [17]:
# ============================================================
# MODEL EVALUATION
# ============================================================

y_pred = model.predict(
    X_test
)

accuracy = accuracy_score(
    y_test_encoded,
    y_pred
)

precision = precision_score(
    y_test_encoded,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test_encoded,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test_encoded,
    y_pred,
    average="weighted",
    zero_division=0
)

print("=" * 80)
print("MODEL PERFORMANCE")
print("=" * 80)

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)

MODEL PERFORMANCE
Accuracy : 0.8674
Precision: 0.8698
Recall   : 0.8674
F1 Score : 0.8680


In [18]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_test_encoded,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

CLASSIFICATION REPORT
                                              precision    recall  f1-score   support

      (vertigo) paroymsal positional vertigo       1.00      1.00      1.00        24
                                        acne       1.00      1.00      1.00        24
                           actinic keratosis       0.75      0.78      0.76       162
                         acute bronchiolitis       0.91      0.93      0.92       241
                            acute bronchitis       0.74      0.71      0.72       243
                          acute bronchospasm       0.50      0.59      0.54       162
                         acute kidney injury       0.97      0.94      0.96       162
                          acute pancreatitis       0.89      0.86      0.88       241
                             acute sinusitis       0.82      0.78      0.80       166
                                        aids       1.00      1.00      1.00        24
                         alcoho

In [19]:
# ============================================================
# LIGHTWEIGHT CROSS VALIDATION
# ============================================================

print("=" * 75)
print("CROSS VALIDATION")
print("=" * 75)

# Use a smaller sample to avoid VS Code/Jupyter crashing
CV_SAMPLE_SIZE = min(20000, len(X_train))

# IMPORTANT:
# Reset index so X and y stay perfectly aligned
cv_data = X_train.copy()
cv_data["__target__"] = y_train.to_numpy()

cv_data = cv_data.sample(
    n=CV_SAMPLE_SIZE,
    random_state=RANDOM_STATE
).reset_index(drop=True)

cv_sample = cv_data.drop(
    columns=["__target__"]
)

cv_sample_y = cv_data["__target__"]

# Keep 100 trees as requested
cv_model = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=2
)

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_scores = cross_val_score(
    cv_model,
    cv_sample,
    cv_sample_y,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=1
)

print("CV sample size:", len(cv_sample))
print("Number of folds:", cv.get_n_splits())
print("CV model trees:", cv_model.n_estimators)

for index, score in enumerate(cv_scores, start=1):
    print(f"Fold {index}: {score:.4f}")

print()
print(f"Mean F1: {cv_scores.mean():.4f}")
print(f"Std F1 : {cv_scores.std():.4f}")

print()
print("Cross-validation completed successfully.")

CROSS VALIDATION
CV sample size: 20000
Number of folds: 3
CV model trees: 100
Fold 1: 0.8883
Fold 2: 0.8788
Fold 3: 0.8860

Mean F1: 0.8844
Std F1 : 0.0041

Cross-validation completed successfully.


In [20]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({

    "Symptom":
        X.columns,

    "Importance":
        model.feature_importances_

})

feature_importance = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 80)
print("TOP 30 IMPORTANT SYMPTOMS")
print("=" * 80)

display(
    feature_importance.head(30)
)

TOP 30 IMPORTANT SYMPTOMS


,Symptom,Importance
0,vomiting,0.017313
1,headache,0.016453
2,nausea,0.012257
3,back pain,0.011701
4,cough,0.009806
5,joint pain,0.009396
6,burning abdominal pain,0.008907
7,fever,0.008866
8,sharp chest pain,0.008844
9,sharp abdominal pain,0.007971


In [21]:
# ============================================================
# FAST HEART RATE TEST
# ============================================================

def prepare_test_input(symptoms):

    input_data = pd.DataFrame(
        0,
        index=[0],
        columns=unified_features,
        dtype=np.int8
    )

    recognized = []
    unknown = []

    for symptom in symptoms:

        normalized = normalize_symptom_name(
            symptom
        )

        if normalized in input_data.columns:

            input_data.loc[
                0,
                normalized
            ] = 1

            recognized.append(
                normalized
            )

        else:

            unknown.append(
                normalize_text(symptom)
            )

    recognized = list(
        dict.fromkeys(recognized)
    )

    unknown = list(
        dict.fromkeys(unknown)
    )

    return (
        input_data,
        recognized,
        unknown
    )


def predict_top_k(symptoms, k=10):

    (
        input_data,
        recognized,
        unknown
    ) = prepare_test_input(symptoms)

    if not recognized:

        raise ValueError(
            "None of the supplied symptoms "
            "exist in the unified feature space."
        )

    probabilities = model.predict_proba(
        input_data
    )[0]

    top_indices = np.argsort(
        probabilities
    )[::-1][:k]

    results = []

    for index in top_indices:

        encoded_class = model.classes_[
            index
        ]

        disease = label_encoder.inverse_transform(
            [encoded_class]
        )[0]

        results.append({

            "Disease":
                disease,

            "Probability":
                float(
                    probabilities[index]
                )

        })

    return (
        pd.DataFrame(results),
        recognized,
        unknown
    )


fast_result, recognized, unknown = predict_top_k(
    ["fast heart rate"],
    k=10
)

print("=" * 80)
print("FAST HEART RATE TEST")
print("=" * 80)

print("Recognized:", recognized)
print("Unknown:", unknown)

fast_display = fast_result.copy()

fast_display["Probability"] = (
    fast_display["Probability"] * 100
).round(2)

display(
    fast_display
)

FAST HEART RATE TEST
Recognized: ['increased heart rate']
Unknown: []


,Disease,Probability
0,heart attack,43.33
1,sinus bradycardia,16.67
2,anxiety,13.33
3,angina,10.00
4,vaginitis,3.33
5,chronic constipation,3.33
6,sebaceous cyst,3.33
7,heart failure,2.83
8,asthma,2.33
9,chronic obstructive pulmonary disease (copd),1.50


In [22]:
# ============================================================
# MULTIPLE SYMPTOM TESTS
# ============================================================

test_cases = {

    "Fast Heart Rate": [
        "fast heart rate"
    ],

    "Thyroid Pattern": [
        "fast heart rate",
        "weight loss",
        "fatigue"
    ],

    "Diabetes Pattern": [
        "polyuria",
        "weight loss",
        "fatigue"
    ],

    "Abdominal Pain": [
        "abdominal pain"
    ],

    "Pneumonia Pattern": [
        "cough",
        "shortness of breath",
        "chills"
    ]
}


for test_name, symptoms in test_cases.items():

    print()
    print("=" * 80)
    print(test_name)
    print("=" * 80)

    try:

        result, recognized, unknown = (
            predict_top_k(
                symptoms,
                k=5
            )
        )

        print(
            "Input:",
            symptoms
        )

        print(
            "Recognized:",
            recognized
        )

        print(
            "Unknown:",
            unknown
        )

        result["Probability"] = (
            result["Probability"] * 100
        ).round(2)

        display(result)

    except Exception as error:

        print(
            "TEST ERROR:",
            error
        )


Fast Heart Rate
Input: ['fast heart rate']
Recognized: ['increased heart rate']
Unknown: []


,Disease,Probability
0,heart attack,43.33
1,sinus bradycardia,16.67
2,anxiety,13.33
3,angina,10.00
4,vaginitis,3.33



Thyroid Pattern
Input: ['fast heart rate', 'weight loss', 'fatigue']
Recognized: ['increased heart rate', 'weight loss', 'fatigue']
Unknown: []


,Disease,Probability
0,jaundice,16.67
1,heart attack,16.67
2,diabetes,16.67
3,multiple sclerosis,16.67
4,obstructive sleep apnea (osa),10.00



Diabetes Pattern
Input: ['polyuria', 'weight loss', 'fatigue']
Recognized: ['polyuria', 'weight loss', 'fatigue']
Unknown: []


,Disease,Probability
0,diabetes,50.00
1,obstructive sleep apnea (osa),16.67
2,multiple sclerosis,13.33
3,hypertensive heart disease,10.00
4,jaundice,6.67



Abdominal Pain
Input: ['abdominal pain']
Recognized: ['abdominal pain']
Unknown: []


,Disease,Probability
0,peptic ulcer disease,36.67
1,alcoholic hepatitis,16.67
2,rectal disorder,5.24
3,temporary or benign blood in urine,3.33
4,vaginitis,3.33



Pneumonia Pattern
Input: ['cough', 'shortness of breath', 'chills']
Recognized: ['cough', 'shortness of breath', 'chills']
Unknown: []


,Disease,Probability
0,pneumonia,63.33
1,sepsis,30.00
2,common cold,3.33
3,acute bronchiolitis,2.22
4,croup,1.11


In [23]:
# ============================================================
# FINAL MODEL VALIDATION
# ============================================================

MODEL_FEATURE_COUNT = getattr(
    model,
    "n_features_in_",
    None
)

MODEL_CLASS_COUNT = len(
    model.classes_
)

ENCODER_CLASS_COUNT = len(
    label_encoder.classes_
)

print("=" * 80)
print("FINAL MODEL VALIDATION")
print("=" * 80)

print(
    "Model type:",
    type(model).__name__
)

print(
    "Model features:",
    MODEL_FEATURE_COUNT
)

print(
    "Feature names:",
    len(unified_features)
)

print(
    "Model classes:",
    MODEL_CLASS_COUNT
)

print(
    "Encoder classes:",
    ENCODER_CLASS_COUNT
)

print(
    "Dataset diseases:",
    y.nunique()
)

if MODEL_FEATURE_COUNT != len(
    unified_features
):

    raise ValueError(
        "Model feature count does not match "
        "unified feature names."
    )

if MODEL_CLASS_COUNT != (
    ENCODER_CLASS_COUNT
):

    raise ValueError(
        "Model class count does not match "
        "label encoder."
    )

if MODEL_CLASS_COUNT != y.nunique():

    raise ValueError(
        "Model class count does not match "
        "unified dataset disease count."
    )

print()
print("FINAL VALIDATION PASSED.")

FINAL MODEL VALIDATION
Model type: RandomForestClassifier
Model features: 342
Feature names: 342
Model classes: 133
Encoder classes: 133
Dataset diseases: 133

FINAL VALIDATION PASSED.


In [24]:
# ============================================================
# SAVE FINAL MODEL ARTIFACTS
# ============================================================

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "disease_model.pkl"
)

FEATURE_PATH = os.path.join(
    MODEL_DIR,
    "feature_names.pkl"
)

ENCODER_PATH = os.path.join(
    MODEL_DIR,
    "label_encoder.pkl"
)

print("=" * 80)
print("SAVING FINAL MODEL")
print("=" * 80)

joblib.dump(
    model,
    MODEL_PATH
)

joblib.dump(
    list(unified_features),
    FEATURE_PATH
)

joblib.dump(
    label_encoder,
    ENCODER_PATH
)

print("Model saved:")
print(MODEL_PATH)

print("Features saved:")
print(FEATURE_PATH)

print("Encoder saved:")
print(ENCODER_PATH)

SAVING FINAL MODEL
Model saved:
c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System\models\disease_model.pkl
Features saved:
c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System\models\feature_names.pkl
Encoder saved:
c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System\models\label_encoder.pkl


In [25]:
# ============================================================
# VERIFY SAVED ARTIFACTS
# ============================================================

for path in [
    MODEL_PATH,
    FEATURE_PATH,
    ENCODER_PATH
]:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Artifact was not created:\n{path}"
        )

    print(
        os.path.basename(path),
        "->",
        os.path.getsize(path),
        "bytes"
    )


# Reload everything
loaded_model = joblib.load(
    MODEL_PATH
)

loaded_features = joblib.load(
    FEATURE_PATH
)

loaded_encoder = joblib.load(
    ENCODER_PATH
)


if len(loaded_features) != (
    loaded_model.n_features_in_
):

    raise ValueError(
        "Saved feature file is incompatible "
        "with saved model."
    )


if len(loaded_encoder.classes_) != (
    len(loaded_model.classes_)
):

    raise ValueError(
        "Saved encoder is incompatible "
        "with saved model."
    )


print()
print("=" * 80)
print("SAVED MODEL VERIFICATION PASSED")
print("=" * 80)

print(
    "Features:",
    len(loaded_features)
)

print(
    "Diseases:",
    len(loaded_encoder.classes_)
)

print(
    "Model:",
    type(loaded_model).__name__
)

disease_model.pkl -> 858655921 bytes
feature_names.pkl -> 6444 bytes
label_encoder.pkl -> 3060 bytes

SAVED MODEL VERIFICATION PASSED
Features: 342
Diseases: 133
Model: RandomForestClassifier


In [26]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 80)
print("FINAL UNIFIED MODEL SUMMARY")
print("=" * 80)

print(
    "Dataset 1 records:",
    len(df1)
)

print(
    "Dataset 3 records:",
    len(df3)
)

print(
    "Combined records:",
    len(X)
)

print(
    "Dataset 1 diseases:",
    len(diseases_1)
)

print(
    "Dataset 3 diseases:",
    len(diseases_3)
)

print(
    "Common diseases:",
    len(common_diseases)
)

print(
    "Final disease classes:",
    len(label_encoder.classes_)
)

print(
    "Final symptom features:",
    len(unified_features)
)

print(
    "Training records:",
    len(X_train)
)

print(
    "Testing records:",
    len(X_test)
)

print(
    "Accuracy:",
    f"{accuracy:.4f}"
)

print(
    "Precision:",
    f"{precision:.4f}"
)

print(
    "Recall:",
    f"{recall:.4f}"
)

print(
    "F1:",
    f"{f1:.4f}"
)

print(
    "CV Mean F1:",
    f"{cv_scores.mean():.4f}"
)

print()
print("FINAL FILES:")
print("1. disease_model.pkl")
print("2. feature_names.pkl")
print("3. label_encoder.pkl")

print()
print("=" * 80)
print("UNIFIED MODEL TRAINING COMPLETED SUCCESSFULLY")
print("=" * 80)

FINAL UNIFIED MODEL SUMMARY
Dataset 1 records: 4920
Dataset 3 records: 96088
Combined records: 101008
Dataset 1 diseases: 41
Dataset 3 diseases: 100
Common diseases: 8
Final disease classes: 133
Final symptom features: 342
Training records: 80806
Testing records: 20202
Accuracy: 0.8674
Precision: 0.8698
Recall: 0.8674
F1: 0.8680
CV Mean F1: 0.8844

FINAL FILES:
1. disease_model.pkl
2. feature_names.pkl
3. label_encoder.pkl

UNIFIED MODEL TRAINING COMPLETED SUCCESSFULLY
